# proof-engine quick start

Compare a down-only mortgage with a standard fixed-rate mortgage across simulated interest-rate paths.

**Every number in this notebook is an arbitrary example input**, not any lender's rates or terms. Outputs are model estimates for hypothetical loans, not an offer of credit.

In [ ]:
from proof_engine import (DownOnlyLoan, HullWhiteMarket, PrepaymentModel, RefinancePolicy, StandardLoan,
                          par_spread_down_only, par_spread_standard, simulate)

market = HullWhiteMarket(flat_rate=3.7, mean_reversion=0.11, volatility=1.3, index_tenor_years=10)
paths = market.simulate(months=360, days_per_month=21, paths=1_000, seed=21, antithetic=True)

down_only = DownOnlyLoan(rate_spread=1.6, floor=2.2, first_step=0.35, subsequent_step=0.6, check_every_days=1)

## Pricing lens: the cost of the down-only feature

Solve each loan's spread over the index at which its cash flows price at par, then take the difference.

In [ ]:
prepayment = PrepaymentModel(turnover_cpr=0.08, refi_max_cpr=0.55, refi_midpoint=0.9, refi_width=0.28, max_total_cpr=0.9)
kw = dict(term_months=360, coupon_strip=0.55, oas=0.4, search=(-2.0, 6.0))

standard_spread = par_spread_standard(paths, prepayment=prepayment, **kw)
down_only_spread = par_spread_down_only(paths, loan=down_only, turnover_cpr=0.08, **kw)
print(f"standard par spread:  {standard_spread:.3f}")
print(f"down-only par spread: {down_only_spread:.3f}")
print(f"feature cost:         {down_only_spread - standard_spread:.3f} points")

## Borrower-cost lens

For one borrower, compare what each loan costs over ten years under three refinancing habits, with the down-only loan priced at its par spread.

In [ ]:
borrowers = {
    "never refinances": RefinancePolicy.never(),
    "refinances late": RefinancePolicy(threshold=1.1, delay_months=8, monthly_probability=0.17, requalify_probability=0.85, closing_costs=0.025),
    "refinances promptly": RefinancePolicy(threshold=0.6, delay_months=0, monthly_probability=1.0, requalify_probability=1.0, closing_costs=0.025),
}
results = {
    name: simulate(paths, balance=400_000, term_years=30, horizon_years=10,
                   down_only=down_only.with_spread(down_only_spread),
                   standard=StandardLoan(rate_spread=standard_spread, refinance=policy),
                   discount_rate=3.1, annual_turnover=0.0, seed=5)
    for name, policy in borrowers.items()
}
for name, r in results.items():
    s = r.summary()
    print(f"{name:>20}: mean {s['savings_mean']:>10,.0f}   p5 {s['savings_p5']:>10,.0f}   p95 {s['savings_p95']:>10,.0f}   P(down-only cheaper) {s['prob_down_only_cheaper']:.2f}")

Positive savings mean the down-only loan cost less on that path. Look at the whole range: there are paths where it costs more.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for name, r in results.items():
    ax.hist(r.savings, bins=80, histtype="step", linewidth=1.5, label=name)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Present-value cost difference (standard minus down-only)")
ax.set_ylabel("Paths")
ax.set_title("Distribution of outcomes (arbitrary example inputs)")
ax.legend()
plt.show()